# Agent
## Agent创建方式
- `create_agent`创建，底层是LangGraph，常见参数：
    - model: 模型
    - name: agent名称。便于追踪、多Agent协作
    - system_prompt: 系统提示词
- 能够自动重试
- invoke消息结构
  ```python
    messages = {
        "messages": []
    }
  ```

## LangChain内置工具的使用
- LangChain内置工具列表：https://docs.langchain.com/oss/python/integrations/tools
- 设置系统提示词
- 设置工具列表（<=5个数量最佳）

In [1]:
import os

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage
from ollama import web_search
from rich import print as rprint
from langchain_tavily import TavilySearch

from common import init_simple_dashscope_model

# 内置工具

web_search = TavilySearch(
    tavily_api_key=os.getenv('TAILLY_API_KEY'),
    max_results=2,
)

agent = create_agent(
    model=init_simple_dashscope_model('qwen-max'),
    tools=[web_search]
)

messages = {
    'messages': [
        SystemMessage(content="你是一个网页信息收集助手"),
        HumanMessage(content="https://lilianweng.github.io/posts/2023-06-23-agent/ 告诉我这个网站里讲了什么内容？")
    ]
}

resp = agent.invoke(messages)
rprint(resp)

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={'tavily_api_key': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

## 自主重试

In [ ]:
from langchain_core.tools import tool
from common import init_simple_dashscope_model
from rich import print as rprint

retry = 0


@tool(description="获取实时天气")
def get_weather(city: str):
    """
    获取实时天气

    Args:
        city: 城市

    Returns:
        城市天气信息
    """

    global retry
    retry += 1
    if retry < 3:
        return "TOOL_UNAVAILABLE:服务不可用"
    return f'{city}今天下雨，温度25度'


agent = create_agent(
    model=init_simple_dashscope_model('qwen-max'),
    tools=[get_weather]
)

messages = {
    'messages': [
        SystemMessage(content="你是一个天气助手，当工具返回TOOL_UNAVAILABLE时表示调用失败，你需要重试，直至拿到结果"),
        HumanMessage(content="深圳今天的天气怎么样？")
    ]
}

resp = agent.invoke(messages)
rprint(resp)

## 结构化输出
通过`response_format`指定
### ProviderStrategy: 需要模型原生支持
### ToolStrategy: (推荐)大都数模型都支持。但深度思考模型不能使用它
    - tool_message_content: ToolStrategy会把Schema注册为伪工具，最终模型输出的结构化数据取决的function_calls的args，
    默认返回的ToolMessage:Returning structured response:xxx，只是历史对话占位。因此默认工具会返回结构化数据，
    当数据复杂冗长时，会消耗更多token，可以用该参数指定简短内容输出。
    - handle_errors: 异常处理策略
        - True: 默认。将异常信息当做ToolMessage告知给大模型，大模型根据错误自主决策
        - False: 遇到异常会程序终止
        - Error元组: 设置为指定异常类型。(MultipleStructuredOutputsError,StructuredOutputValidationError)等。如果出现了额外的异常，程序终止
        - str: 异常时，返回自定义文本给大模型。但注意，如果内容不具备异常说明，模型无法发现异常，导致大模型无法自主决策，循环调用，浪费Token
        - callable: 自定义异常回调，用于精细化处理
### AutoStrategy: 自动
### Type: 直接写类型（已弃用，不推荐）
### Union: 联合类型，可以同时指定多个数据结构，LLM会选择最合适的(互斥，必须是多选一)
    - 如果一个要求同时返回多个类型，需要使用嵌套。
    如果使用Union，但如果提示词要求同时返回多个类型，这违反了Union的唯一约束，
    LangChain会报MultipleStructuredOutputsError（多返回的工具调用数>1），
    ToolStrategy默认handle_errors=True，所以错误信息会当做ToolMessage发给LLM，
    LLM在发现错误后，能够重试，最终给出唯一结果

In [ ]:
from common import init_simple_dashscope_model
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage
from rich import print as rprint
from typing import Union




class Movie(BaseModel):
    title: str = Field(
        description="电影标题",
    )
    rating: float = Field(
        description="评分"
    )


class MovieList(BaseModel):
    movie_list: list[Movie] = Field(
        description="推荐电影列表"
    )

class CustomerComment(BaseModel):
    customer_name: str = Field(
        description="用户名称"
    )
    content: str = Field(
        description="评论内容"
    )


# extra_body = {
#     "enable_thinking": False  # 启用思考模式
# }
# model = init_simple_dashscope_model('qwen3.7-plus', extra_body)
model = init_simple_dashscope_model('qwen-max')
agent = create_agent(
    model=model,
    # response_format=ProviderStrategy(Movie),
    # response_format=ToolStrategy(MovieList),
    response_format=ToolStrategy(Union[MovieList, CustomerComment]),
    system_prompt="""
    你是一个影音推荐助手
    """
)

resp = agent.invoke(
    {
        'messages': [
            HumanMessage('给我推荐三部历史最经典的电影，并给我看一下用户的经典好评')
            # HumanMessage('教父的经典好评')
        ]
    }
)

rprint(resp)

## 流式输出
### 7种流式输出模式
    (每种模式输出结果的结构都不一样)
    - values：递进全量消息列表。每一步拍一张全景照
    - updates：增量消息。每一步只显示增加的消息
    - messages：打字机效果， 返回一个元组，获取消息内容：message[0].content
    - tasks：记录了一个id。能记录节点的起止与错误
    - debug：在tasks的基础上，加了step和timestamp。等价于tasks + checkpoints,适合开发调试
    - checkpoints: 可以设置状态检查点，适合状态持久化、断点续传、状态演进
    - custom: 可以writer任意数据，可以自定义输出流，比如进度。只订阅该模式，只会输出自定义流，
    如果需要同时查看多种内容，需要订阅多个。

In [16]:
import time
from langgraph.config import get_stream_writer
from langchain_core.tools import tool


@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"


In [ ]:
from common import init_simple_dashscope_model
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage
from rich import print as rprint
from typing import Union

from langgraph.checkpoint.memory import InMemorySaver


class Movie(BaseModel):
    title: str = Field(
        description="电影标题",
    )
    rating: float = Field(
        description="评分"
    )


class MovieList(BaseModel):
    movie_list: list[Movie] = Field(
        description="推荐电影列表"
    )

class CustomerComment(BaseModel):
    customer_name: str = Field(
        description="用户名称"
    )
    content: str = Field(
        description="评论内容"
    )

# checkpoints mode
checkpointer = InMemorySaver()


# model = init_simple_dashscope_model('qwen3.7-plus', extra_body)
model = init_simple_dashscope_model('qwen-max')
agent = create_agent(
    model=model,
    # response_format=ProviderStrategy(Movie),
    # response_format=ToolStrategy(MovieList),
    response_format=ToolStrategy(Union[MovieList, CustomerComment]),
    tools=[generate_sales_report, generate_inventory_report],
    system_prompt="""
    你是一个影音推荐助手
    """,
    # checkpoints mode
    # checkpointer=checkpointer
)

# checkpoints mode
config = {
    "configurable": {
        "thread_id": "session_01"
    }
}

# checkpoints mode
checkpoint_count = 0

resp = agent.stream(
    {
        'messages': [
            # HumanMessage('给我推荐三部历史最经典的电影，并给我看一下用户的经典好评')
            HumanMessage('生成销售报告和库存报告')
            # HumanMessage('教父的经典好评')
        ]
    },
    # stream_mode='checkpoints',
    # config=config,
    # stream_mode='custom',
    # 订阅多个
    stream_mode=['custom', 'updates'],
)

total = 20


for chunk in resp:
    if total < 0:
        break
    ##  checkpoints mode
    # checkpoint_count += 1
    # rprint(f'current checkpoint: {checkpoint_count}')
    rprint(chunk)
    rprint('-' * 50)
    # total-=1